In [0]:
#Imports
from pyspark.sql.functions import from_json,col,regexp_replace, trim,explode
from pyspark.sql.functions import split
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

#Loading Table  

In [0]:
Games=spark.table("steam_data.default.steam_games")

Review=spark.table("steam_data.default.review")

number of rows and columns

In [0]:
def table_size(df, name):
    data = [(name, df.count(), len(df.columns))]
    spark.createDataFrame(data, ["Table", "Rows", "Cols"]).show()
table_size(Games, "Games")
table_size(Review, "Review")

#Drop Duplicates

In [0]:
Games=Games.dropDuplicates()
Review=Review.dropDuplicates()

print("AFTER DROPPING DUPLICATES")

table_size(Games,"Games")
table_size(Review,"Reviews")




**DUPLICATES NOT FOUND AS THE SIZE IS UNCHANGED AFTER DROPPING DUPLICATES**

#Games Data

Counting the number of null in each columns

In [0]:

null_count={}
for colu in Games.columns:   
    number_of_null = Games.filter(col(colu).isNull()).count()
    null_count[colu]=number_of_null


In [0]:

null_count_df = spark.createDataFrame(null_count.items(), ["column", "null_count"])
null_count_df.show()


In [0]:
Games.columns

As there are 42 columns , I am keeping only the ones which are useful for the analysis

In [0]:
columns_to_keep=['app_id','name','price','price_status','estimated_owners','genres','categories','positive','negative','recommendations','peak_ccu','metacritic_score','user_score','average_playtime_forever','windows','mac','linux','supported_Languages']
Games=Games.select(columns_to_keep)
Games=Games.withColumnRenamed("supported_Languages","supported_languages")
Games=Games.withColumn("price",Games["price"].cast("double"))

In [0]:
Games.describe().show()

display(Games)

We can see the estimated_owners column in given as a-b or a..b, we can split it into a,b and store (a+b)/2. As there are extra spaces we will use trim to remove them

In [0]:
Games = Games.withColumn(
    "estimated_owners",
    split(col("estimated_owners"), r"\.\.|\s*-\s*")
)

In [0]:
Games = Games.withColumn(
    "estimated_owners",
    (
        regexp_replace(trim(col("estimated_owners").getItem(0)), ",", "").cast("double")
        +
        regexp_replace(trim(col("estimated_owners").getItem(1)), ",", "").cast("double")
    ) / 2
)

In [0]:
Games.select("estimated_owners").show()

Store this as a new table

In [0]:
Games.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("steam_data.default.silver_games")

More processing on the new table

In [0]:
silver_games=spark.table("steam_data.default.silver_games")
silver_games.describe().show()
display(silver_games)

#Handling Genre and Category

In [0]:
genre_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("description", StringType(), True)
    ])
)

silver_games = silver_games.withColumn(
    "genres_parsed",
    from_json(col("genres"), genre_schema)
)

In [0]:
game_genre_df = (
    silver_games
    .select(
        "app_id",
        explode(col("genres_parsed")).alias("genres")
    )
    .select(
        "app_id",
        col("genres.id").alias("genre_id")
    )
)

In [0]:
game_genre_df.show()

In [0]:
genres_df = (
    silver_games
    .select(explode(col("genres_parsed")).alias("genres"))
    .select(
        col("genres.id").alias("genre_id"),
        col("genres.description").alias("description")
    )
    .dropDuplicates(["genre_id"])
)

In [0]:
genres_df.show()